# Mixture of Experts from Scratch

This notebook is a teaching-first walkthrough of a Mixture of Experts (MoE) language model built in PyTorch.
We build every MoE-specific component from scratch and explain why each design choice exists.

The core idea behind MoE is deceptively simple: instead of one big feed-forward network, use many small "expert" networks and a learned router that picks the best ones for each token.
This lets you build a model with enormous total capacity while only activating a small fraction of the parameters per token, giving you more knowledge without proportional compute cost.

## Audience
- Learners who have already worked through the simple transformer and Llama notebooks in this series.
- Students who understand attention, residual connections, and feed-forward networks.
- Anyone who wants to understand how models like DeepSeek V3 (671B total / 37B active) achieve their scale.

## Prerequisites
- The Llama notebook in this series (for RMSNorm, RoPE, GQA, and SwiGLU).
- Comfort with PyTorch modules, tensors, and basic training loops.
- A rough idea of how transformer blocks work.

## Learning Goals
By the end of this notebook, you should be able to:
1. Explain how a top-k router selects experts for each token and why the routing weights are normalized.
2. Describe the load-balancing auxiliary loss and why training collapses without it.
3. Understand the role of a shared expert that processes every token regardless of routing.
4. Distinguish between total parameters and active parameters in a sparse model.
5. Build a hybrid dense + MoE transformer where the first layers are dense and the rest use routing.

## Outline

1. Set up the environment and imports.
2. Define the `MoEConfig` with both standard and MoE-specific fields.
3. Recap shared building blocks from Llama (RMSNorm, RoPE, GQA, SwiGLU).
4. Build MoE-specific components one at a time:
   - Expert MLP
   - Top-K Router (with load-balancing loss)
   - MoE Layer (router + experts + shared expert)
   - Dense Block vs MoE Block
5. Assemble the full `MoEModel` (hybrid dense prefix + MoE layers).
6. Smoke test the model (total vs active params, architecture verification).
7. Build the data pipeline and verify it.
8. Add training utilities (with auxiliary loss logging).
9. Run a lightweight pipeline preview.
10. Add text generation.
11. Optional checkpoint demo.
12. Exercises and extensions.
13. Common pitfalls and next steps.

## 1. Setup

This install cell keeps the notebook easy to run in a fresh environment such as Colab.
If your environment already has these packages, rerunning this cell is harmless.

In [ ]:
!pip install torch numpy requests -q

## 2. Imports

We gather the core tools once so every later cell can stay focused on one idea.

Teaching note:
- `dataclass` gives us a clean config object.
- `Path` helps keep file paths portable between local Jupyter and Colab-style environments.
- `torch.nn` holds reusable neural network layers.
- `torch.nn.functional` contains tensor-level math functions like `softmax` and `cross_entropy`.

In [ ]:
import math
import os
import time
from dataclasses import dataclass
from pathlib import Path

import requests
import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. Configuration

### `MoEConfig`

This dataclass collects the high-level knobs that control the model.
It extends the Llama configuration with several MoE-specific fields.

Standard transformer fields (same as Llama):
- `vocab_size=256` because we tokenize raw UTF-8 bytes.
- `n_embd` is the width of the hidden representation.
- `n_head` and `n_kv_head` control Grouped Query Attention (GQA).
- `n_layer` controls total depth (dense + MoE layers combined).
- `seq_len` is the maximum context window.
- `dropout=0.0` keeps the tutorial deterministic.
- `rope_theta` is the base frequency for Rotary Position Embeddings.

MoE-specific fields (the new ones):
- `n_dense_layers=1`: the first N layers use a standard MLP instead of routing. These dense prefix layers build basic token representations before routing makes sense. DeepSeek V3 and Llama 4 both use this hybrid approach.
- `n_experts=8`: total number of expert MLPs available. More experts means more total capacity.
- `n_experts_active=2`: how many experts each token passes through (top-k). This is the "sparsity knob" -- it controls the ratio of active to total parameters.
- `has_shared_expert=True`: whether to include an always-on expert that processes every token (DeepSeek V3 innovation).
- `aux_loss_weight=0.01`: how strongly the load-balancing loss pushes toward uniform expert utilization.

In [ ]:
@dataclass
class MoEConfig:
    vocab_size: int = 256
    n_embd: int = 64
    n_head: int = 4
    n_kv_head: int = 2
    n_layer: int = 4            # total layers (dense + MoE)
    n_dense_layers: int = 1     # first N layers are dense (no routing)
    seq_len: int = 256
    dropout: float = 0.0
    rope_theta: float = 10000.0

    # MoE specific
    n_experts: int = 8          # total number of expert MLPs
    n_experts_active: int = 2   # top-k experts activated per token
    has_shared_expert: bool = True  # always-on shared expert (DeepSeek style)
    aux_loss_weight: float = 0.01   # weight of load-balancing loss

config = MoEConfig()
print(config)

## 4. Shared Building Blocks (Recap from Llama)

The MoE architecture reuses several components from Llama-style transformers.
We include them here so this notebook is fully self-contained, but the explanations are brief recaps.
If any of these feel unfamiliar, work through the Llama notebook first.

### `RMSNorm`

A simpler alternative to LayerNorm that rescales each vector by its root-mean-square magnitude.
It keeps activations numerically stable and appears in most modern transformer designs.

Formula: `RMSNorm(x) = x / sqrt(mean(x^2) + eps) * gamma`

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return (x / rms) * self.weight

### RoPE (Rotary Position Embeddings)

Instead of adding learned position vectors, RoPE encodes position by rotating query and key vectors.
The angle of rotation depends on the position, so the dot product Q*K naturally captures relative distance.
This helps the model generalize to unseen sequence lengths.

Two functions work together:
- `precompute_rope_freqs` builds cosine and sine tables once.
- `apply_rope` rotates Q and K tensors using those tables.

The frequency formula `theta_i = base^(-2i/d)` creates a multi-scale encoding:
early dimensions rotate fast (local patterns), later dimensions rotate slowly (global patterns).

In [ ]:
def precompute_rope_freqs(head_dim: int, seq_len: int, theta: float = 10000.0):
    """Precompute the complex exponentials for RoPE.

    Returns cos and sin tensors of shape (seq_len, head_dim // 2).
    """
    freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    t = torch.arange(seq_len).float()
    angles = torch.outer(t, freqs)  # (seq_len, head_dim // 2)
    return torch.cos(angles), torch.sin(angles)


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """Apply rotary embeddings to a tensor.

    Args:
        x: shape (batch, n_head, seq_len, head_dim)
        cos, sin: shape (seq_len, head_dim // 2), precomputed frequencies
    """
    T = x.shape[2]
    x_pairs = x.float().reshape(*x.shape[:-1], -1, 2)
    x0 = x_pairs[..., 0]  # even dimensions
    x1 = x_pairs[..., 1]  # odd dimensions

    cos_t = cos[:T].unsqueeze(0).unsqueeze(0)  # (1, 1, T, head_dim//2)
    sin_t = sin[:T].unsqueeze(0).unsqueeze(0)  # (1, 1, T, head_dim//2)

    out0 = x0 * cos_t - x1 * sin_t
    out1 = x0 * sin_t + x1 * cos_t

    out = torch.stack([out0, out1], dim=-1).flatten(-2)
    return out.type_as(x)

### `GroupedQueryAttention` (GQA)

GQA shares key and value projections across groups of query heads.
With `n_head=4` and `n_kv_head=2`, each pair of query heads shares one set of keys and values.
This cuts KV-cache memory during inference while keeping most of the attention quality.

Special cases:
- `n_kv_head == n_head`: standard multi-head attention (no sharing).
- `n_kv_head == 1`: multi-query attention (maximum sharing).

In [ ]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, config: MoEConfig, cos: torch.Tensor, sin: torch.Tensor):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        assert config.n_head % config.n_kv_head == 0

        self.n_head = config.n_head
        self.n_kv_head = config.n_kv_head
        self.n_groups = config.n_head // config.n_kv_head
        self.head_dim = config.n_embd // config.n_head

        self.W_q = nn.Linear(config.n_embd, config.n_head * self.head_dim, bias=False)
        self.W_k = nn.Linear(config.n_embd, config.n_kv_head * self.head_dim, bias=False)
        self.W_v = nn.Linear(config.n_embd, config.n_kv_head * self.head_dim, bias=False)
        self.W_o = nn.Linear(config.n_embd, config.n_embd, bias=False)

        self.dropout = nn.Dropout(config.dropout)

        self.register_buffer("rope_cos", cos)
        self.register_buffer("rope_sin", sin)

        causal_mask = torch.tril(torch.ones(config.seq_len, config.seq_len))
        self.register_buffer("causal_mask", causal_mask.view(1, 1, config.seq_len, config.seq_len))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape

        q = self.W_q(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        q = apply_rope(q, self.rope_cos, self.rope_sin)
        k = apply_rope(k, self.rope_cos, self.rope_sin)

        if self.n_groups > 1:
            k = k.repeat_interleave(self.n_groups, dim=1)
            v = v.repeat_interleave(self.n_groups, dim=1)

        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn = attn.masked_fill(self.causal_mask[:, :, :T, :T] == 0, float("-inf"))
        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)

        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.W_o(out)
        return out

### `SwiGLU`

The gated MLP used in Llama-style transformers.
Instead of a simple `up -> activation -> down` path, SwiGLU uses two parallel projections:
`SwiGLU(x) = Swish(W_gate * x) * (W_up * x)`, then projects back down.

The gate learns *what information to let through*, while the up-projection learns *what information to compute*.
This consistently outperforms standard activations like ReLU or GELU.

Because we have two up-projections instead of one, the hidden dimension is reduced to `4 * n_embd * 2/3` to keep parameter count comparable.

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, config: MoEConfig):
        super().__init__()
        hidden_dim = int(4 * config.n_embd * 2 / 3)
        hidden_dim = ((hidden_dim + 7) // 8) * 8  # round to nearest multiple of 8

        self.w_gate = nn.Linear(config.n_embd, hidden_dim, bias=False)
        self.w_up = nn.Linear(config.n_embd, hidden_dim, bias=False)
        self.w_down = nn.Linear(hidden_dim, config.n_embd, bias=False)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gate = F.silu(self.w_gate(x))
        up = self.w_up(x)
        x = gate * up
        x = self.w_down(x)
        x = self.dropout(x)
        return x

## 5. MoE-Specific Building Blocks

This is the heart of the notebook.
We now build the components that make Mixture of Experts work, one at a time.
The order mirrors how the final architecture is assembled:
1. a single expert MLP,
2. a router that picks which experts to use,
3. the MoE layer that combines routing with expert computation,
4. the two types of transformer blocks (dense vs MoE).

### `Expert`

Each expert is a small SwiGLU MLP.
In a real MoE model, all experts share the same architecture but learn different specializations through routing.
Some experts might become good at code, others at math, others at dialogue.

The architecture is identical to SwiGLU, but we define it as a separate class for clarity.
In the MoE layer, we will create `n_experts` of these (8 in our config) and a router will decide which ones each token passes through.

Note that each expert is a full MLP on its own -- the "mixture" does not mean we blend internal expert weights.
Instead, we run the token through the selected experts independently and combine their *outputs*.

In [ ]:
class Expert(nn.Module):
    """A single expert MLP (same architecture as SwiGLU)."""
    def __init__(self, config: MoEConfig):
        super().__init__()
        hidden_dim = int(4 * config.n_embd * 2 / 3)
        hidden_dim = ((hidden_dim + 7) // 8) * 8

        self.w_gate = nn.Linear(config.n_embd, hidden_dim, bias=False)
        self.w_up = nn.Linear(config.n_embd, hidden_dim, bias=False)
        self.w_down = nn.Linear(hidden_dim, config.n_embd, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))

### `TopKRouter`

This is the most important new component in MoE.
The router is a single linear layer that scores every expert for every token, then picks the top-k.

**How routing works, step by step:**
1. A linear layer maps each token from `n_embd` dimensions to `n_experts` scores (one score per expert).
2. We take the top-k scores (k=2 in our config). This is the "sparse" part -- most experts are ignored.
3. We softmax the top-k scores so they sum to 1. These become the mixing weights for expert outputs.

**Why load balancing matters:**

Without an auxiliary loss, the router quickly learns to send all tokens to the same 1-2 experts.
Why? If expert A happens to get slightly more training signal early on, it gets better, so the router sends even more tokens to it, so it gets even better -- a positive feedback loop.
The result is that 6 out of 8 experts sit idle with wasted parameters.

**The load-balancing auxiliary loss:**

The auxiliary loss has two ingredients:
- `f_i` = fraction of tokens routed to expert i (how many tokens actually go there)
- `p_i` = average router probability for expert i (how likely the router thinks each expert is, across all tokens)

The loss is: `N * sum(f_i * p_i)` where N is the number of experts.

Intuition: this product `f_i * p_i` is minimized when load is perfectly balanced (`f_i = p_i = 1/N` for all experts).
If one expert gets too much traffic (high `f_i`) AND the router assigns it high probability (high `p_i`), the product is large and the loss pushes back.
The `N` multiplier keeps the loss scale consistent regardless of expert count.

This loss is added to the main cross-entropy loss with a small weight (0.01), so it gently nudges the router without dominating the language modeling objective.

In [ ]:
class TopKRouter(nn.Module):
    def __init__(self, config: MoEConfig):
        super().__init__()
        self.n_experts = config.n_experts
        self.top_k = config.n_experts_active

        # Router: maps each token to expert scores
        self.gate = nn.Linear(config.n_embd, config.n_experts, bias=False)

    def forward(self, x: torch.Tensor):
        """
        Args:
            x: (batch * seq_len, n_embd) -- flattened token representations

        Returns:
            router_weights: (batch * seq_len, top_k) -- normalized weights for selected experts
            expert_indices: (batch * seq_len, top_k) -- which experts were selected
            aux_loss: scalar -- load balancing loss
        """
        # Score each expert for each token
        logits = self.gate(x)  # (N_tokens, n_experts)

        # Select top-k experts per token
        top_k_logits, expert_indices = logits.topk(self.top_k, dim=-1)
        # Normalize top-k scores to sum to 1
        router_weights = F.softmax(top_k_logits, dim=-1)

        # --- Load-balancing auxiliary loss ---
        # f_i: fraction of tokens where expert i is in the top-k
        mask = F.one_hot(expert_indices, self.n_experts).sum(dim=1).float()  # (N_tokens, n_experts)
        f = mask.mean(dim=0)  # fraction of tokens per expert

        # p_i: average router probability per expert (using full softmax, not just top-k)
        p = F.softmax(logits, dim=-1).mean(dim=0)

        # aux_loss = N * sum(f_i * p_i) -- minimized when load is balanced
        aux_loss = self.n_experts * (f * p).sum()

        return router_weights, expert_indices, aux_loss

### `MoELayer`

The MoE layer combines the router with the pool of expert MLPs.
For each token, the process is:
1. The router selects top-k experts and their weights.
2. The token is processed by each selected expert independently.
3. Expert outputs are combined as a weighted sum using the router weights.
4. If a shared expert exists, its output is added on top.

**The expert loop:**

Our implementation loops over experts (not tokens).
For each expert, we find all tokens that selected it, batch them together, run them through the expert, and scatter the weighted outputs back.
This is simpler to understand than the alternative (looping over tokens).

**Why a shared expert?**

DeepSeek V3 introduced the idea of an "always-on" expert that processes every token regardless of routing.
The intuition: some patterns are universal (basic grammar, common words, punctuation handling).
Routing these through the competitive top-k system is wasteful -- they would just eat up expert slots that could be used for specialized knowledge.
The shared expert captures these common patterns, freeing the routed experts to specialize in rarer, more interesting patterns.

**Implementation note:**

Production MoE models use grouped GEMM or expert-parallel strategies for efficiency.
Our loop-over-experts approach is much simpler to read but slower.
That is the right trade-off for a teaching notebook.

In [ ]:
class MoELayer(nn.Module):
    def __init__(self, config: MoEConfig):
        super().__init__()
        self.router = TopKRouter(config)
        self.experts = nn.ModuleList([Expert(config) for _ in range(config.n_experts)])

        # Optional shared expert (DeepSeek V3 style)
        self.shared_expert = Expert(config) if config.has_shared_expert else None
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor):
        """
        Args:
            x: (batch, seq_len, n_embd)

        Returns:
            output: (batch, seq_len, n_embd)
            aux_loss: scalar load-balancing loss
        """
        B, T, C = x.shape
        # Flatten to (B*T, C) for routing -- each token routed independently
        x_flat = x.view(-1, C)

        # Route tokens to experts
        router_weights, expert_indices, aux_loss = self.router(x_flat)
        # router_weights: (B*T, top_k)
        # expert_indices: (B*T, top_k)

        # Compute expert outputs
        # For each expert, find which tokens selected it, process them, weight and accumulate
        output = torch.zeros_like(x_flat)

        for i, expert in enumerate(self.experts):
            # Find (token_idx, slot_idx) pairs where this expert was selected
            token_idx, slot_idx = torch.where(expert_indices == i)

            if token_idx.numel() == 0:
                continue  # no tokens routed to this expert

            # Gather the tokens assigned to this expert
            expert_input = x_flat[token_idx]  # (n_assigned, C)

            # Process through expert
            expert_output = expert(expert_input)  # (n_assigned, C)

            # Weight by router score and accumulate
            weights = router_weights[token_idx, slot_idx].unsqueeze(-1)  # (n_assigned, 1)
            output.index_add_(0, token_idx, expert_output * weights)

        # Add shared expert output (processes ALL tokens, no routing)
        if self.shared_expert is not None:
            output = output + self.shared_expert(x_flat)

        output = self.dropout(output)
        return output.view(B, T, C), aux_loss

### `DenseBlock` vs `MoEBlock`

Real MoE models like DeepSeek V3 and Llama 4 use a hybrid architecture: the first few layers are dense (standard MLP), and the rest are MoE.

**Why start with dense layers?**

The earliest layers in a transformer build basic token representations -- combining byte-level features into something meaningful.
Routing at this stage would be premature because the representations are not yet rich enough for the router to make good decisions about specialization.
Dense prefix layers give every token the same processing to build a solid foundation.

**The interface contract:**

Both block types return `(output, aux_loss)`.
Dense blocks always return `aux_loss = 0.0` because there is no routing.
MoE blocks return the real auxiliary loss from their router.
This uniform interface lets the model loop through all blocks identically and accumulate the total auxiliary loss.

In [ ]:
class DenseBlock(nn.Module):
    """Standard transformer block with SwiGLU MLP (for dense prefix layers)."""
    def __init__(self, config: MoEConfig, cos: torch.Tensor, sin: torch.Tensor):
        super().__init__()
        self.ln_1 = RMSNorm(config.n_embd)
        self.attn = GroupedQueryAttention(config, cos, sin)
        self.ln_2 = RMSNorm(config.n_embd)
        self.mlp = SwiGLU(config)

    def forward(self, x: torch.Tensor):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x, torch.tensor(0.0, device=x.device)  # no aux loss from dense block


class MoEBlock(nn.Module):
    """Transformer block with MoE replacing the MLP."""
    def __init__(self, config: MoEConfig, cos: torch.Tensor, sin: torch.Tensor):
        super().__init__()
        self.ln_1 = RMSNorm(config.n_embd)
        self.attn = GroupedQueryAttention(config, cos, sin)
        self.ln_2 = RMSNorm(config.n_embd)
        self.moe = MoELayer(config)

    def forward(self, x: torch.Tensor):
        x = x + self.attn(self.ln_1(x))
        moe_out, aux_loss = self.moe(self.ln_2(x))
        x = x + moe_out
        return x, aux_loss

## 6. Full MoE Model

### `MoEModel`

This class assembles the complete language model.
The forward pass tells a clear story:
1. Turn token IDs into vectors with an embedding table.
2. Pass through a stack of blocks (dense prefix + MoE layers).
3. Accumulate auxiliary losses from MoE layers.
4. Normalize and project to vocabulary logits.
5. Combine cross-entropy loss with the weighted auxiliary loss.

**Key differences from a standard transformer:**

- **No positional embedding table.** RoPE handles position inside the attention layers.
- **Hybrid block stack.** The first `n_dense_layers` blocks use a standard SwiGLU MLP. The remaining blocks use MoE.
- **Auxiliary loss accumulation.** Each MoE block returns its own load-balancing loss. We average these across MoE layers and add the result (scaled by `aux_loss_weight`) to the cross-entropy loss.

**Total vs active parameters:**

This is one of the most important concepts in MoE.
Our model has 8 experts per MoE layer, but each token only uses 2.
That means 6 experts worth of parameters are "inactive" for any given token.
The `count_parameters` method reports both numbers so you can see the sparsity ratio.

For context, DeepSeek V3 has 671B total parameters but only 37B active per token -- a 18x sparsity ratio.
Our tiny model has a more modest ratio, but the principle is identical.

In [ ]:
class MoEModel(nn.Module):
    def __init__(self, config: MoEConfig):
        super().__init__()
        self.config = config

        self.wte = nn.Embedding(config.vocab_size, config.n_embd)

        # Precompute RoPE frequencies
        head_dim = config.n_embd // config.n_head
        cos, sin = precompute_rope_freqs(head_dim, config.seq_len, config.rope_theta)

        # Build layers: dense prefix + MoE layers
        self.blocks = nn.ModuleList()
        for i in range(config.n_layer):
            if i < config.n_dense_layers:
                self.blocks.append(DenseBlock(config, cos, sin))
            else:
                self.blocks.append(MoEBlock(config, cos, sin))

        self.ln_f = RMSNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # Initialize weights
        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("W_o.weight") or name.endswith("w_down.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.config.seq_len

        x = self.wte(idx)

        # Accumulate auxiliary losses from MoE layers
        total_aux_loss = torch.tensor(0.0, device=idx.device)
        n_moe_layers = 0

        for block in self.blocks:
            x, aux_loss = block(x)
            if aux_loss.item() > 0:
                total_aux_loss = total_aux_loss + aux_loss
                n_moe_layers += 1

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            ce_loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            # Add load-balancing loss (averaged over MoE layers)
            if n_moe_layers > 0:
                avg_aux_loss = total_aux_loss / n_moe_layers
                loss = ce_loss + self.config.aux_loss_weight * avg_aux_loss
            else:
                loss = ce_loss

        return logits, loss

    def count_parameters(self):
        total = sum(p.numel() for p in self.parameters())
        # Count "active" params (what runs per token): exclude inactive expert params
        expert_params = sum(p.numel() for p in self.blocks[-1].moe.experts[0].parameters()) if hasattr(self.blocks[-1], 'moe') else 0
        n_moe_layers = self.config.n_layer - self.config.n_dense_layers
        inactive_per_layer = (self.config.n_experts - self.config.n_experts_active) * expert_params
        active = total - (inactive_per_layer * n_moe_layers)

        print(f"Total parameters:  {total:,}")
        print(f"Active parameters: {active:,} (per token)")
        print(f"  Token embeddings: {self.wte.weight.numel():,}")
        for i, block in enumerate(self.blocks):
            block_params = sum(p.numel() for p in block.parameters())
            kind = "Dense" if i < self.config.n_dense_layers else "MoE"
            print(f"  Block {i} ({kind}): {block_params:,}")
        print(f"  Final norm: {sum(p.numel() for p in self.ln_f.parameters()):,}")
        print(f"  LM head:   {self.lm_head.weight.numel():,}")
        return total

## 7. Model Smoke Test

Before training anything, we want a quick confidence check that the tensor shapes, gradients, and architecture behave as expected.

What this test demonstrates:
- The model accepts integer token IDs and produces logits with shape `(batch, time, vocab_size)`.
- The random-initialization loss is in the right ballpark (near `log(256) = 5.54`, plus a small auxiliary loss contribution).
- Gradients flow through the entire network, including through the router and experts.
- The architecture has the correct pattern: 1 dense block followed by 3 MoE blocks.
- Total parameters are significantly larger than active parameters.
- The shared expert is present in MoE blocks.

In [ ]:
model = MoEModel(config)
model.count_parameters()

# Test forward pass (no targets)
idx = torch.randint(0, config.vocab_size, (2, 32))
logits, _ = model(idx)
print(f"\nLogits shape: {logits.shape}")

# Test with targets (loss includes aux load-balancing loss)
targets = torch.randint(0, config.vocab_size, (2, 32))
logits, loss = model(idx, targets)
print(f"Random-init loss: {loss.item():.4f}")
print(f"Expected rough baseline: {math.log(config.vocab_size):.4f} + small aux loss")

# Test gradient flow
loss.backward()
grad_count = sum(1 for p in model.parameters() if p.grad is not None)
print(f"Parameters with gradients: {grad_count}")

# Verify architecture: first block should be dense, rest should be MoE
assert isinstance(model.blocks[0], DenseBlock), "First block should be dense"
for i in range(1, config.n_layer):
    assert isinstance(model.blocks[i], MoEBlock), f"Block {i} should be MoE"
print(f"\nArchitecture: {config.n_dense_layers} dense + {config.n_layer - config.n_dense_layers} MoE blocks")

# Verify total > active params
total = sum(p.numel() for p in model.parameters())
print(f"Sparsity: {config.n_experts} experts, {config.n_experts_active} active per token")

# Verify shared expert exists
moe_block = model.blocks[-1]
assert moe_block.moe.shared_expert is not None, "Should have shared expert"
print(f"Shared expert: present (DeepSeek V3 style)")

## 8. Data Pipeline

A language model learns from one very long stream of tokens.
Our job is to turn that stream into many smaller `(input, target)` examples.

This section is identical to the simple transformer notebook.
We use raw UTF-8 bytes as our tokenizer, which keeps the modeling ideas front and center.

### Dataset Paths and Constants

Notebooks are often run from different working directories.
This cell finds the project root in a way that works both when the notebook is opened from the repository root and when it is opened from the `notebook/` folder.

We also define the Tiny Shakespeare download URL here so later functions can stay focused on behavior rather than configuration.

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "model.py").exists() and (PROJECT_ROOT.parent / "model.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints" / "moe"
SHAKESPEARE_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

### `encode` and `decode`

`encode` converts text into integer token IDs using raw UTF-8 bytes.
`decode` reverses the process so we can turn generated token IDs back into readable text.

Why start with bytes?
- No tokenizer training step, no vocabulary-building ceremony.
- Easy to inspect and reason about.
- Perfect for teaching the language modeling loop.

Trade-off: byte-level models need longer sequences than subword tokenizers to express the same text.

In [ ]:
def encode(text: str) -> list[int]:
    return list(text.encode("utf-8"))


def decode(tokens: list[int]) -> str:
    return bytes(tokens).decode("utf-8", errors="replace")

### `download_shakespeare`

This function downloads the Tiny Shakespeare dataset once and caches it on disk.
On later runs it simply reads the local file.

In [ ]:
def download_shakespeare() -> str:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    filepath = DATA_DIR / "shakespeare.txt"

    if not filepath.exists():
        print(f"Downloading tiny Shakespeare to {filepath}...")
        response = requests.get(SHAKESPEARE_URL)
        response.raise_for_status()
        filepath.write_text(response.text)
        print(f"Downloaded {len(response.text):,} characters.")

    return filepath.read_text()

### `get_batch` and `prepare_data`

`get_batch` randomly samples starting positions from a token stream, slices out windows of length `seq_len`, and creates the shifted targets for next-token prediction.

`prepare_data` pulls the whole data pipeline together: download, encode, tensorize, and split.

In [ ]:
def get_batch(data: torch.Tensor, batch_size: int, seq_len: int, device: str = "cpu"):
    max_start = len(data) - seq_len - 1
    starts = torch.randint(0, max_start, (batch_size,))
    x = torch.stack([data[i : i + seq_len] for i in starts])
    y = torch.stack([data[i + 1 : i + 1 + seq_len] for i in starts])
    return x.to(device), y.to(device)


def prepare_data(val_fraction: float = 0.1):
    text = download_shakespeare()
    tokens = encode(text)
    data = torch.tensor(tokens, dtype=torch.long)
    split_idx = int(len(data) * (1 - val_fraction))
    train_data = data[:split_idx]
    val_data = data[split_idx:]
    return train_data, val_data

## 9. Data Smoke Test

We test three things here:
1. Tokenizer round-trip correctness.
2. Dataset size after preparation.
3. Shift-by-one behavior in the training batch.

When teaching, these checks are valuable because many training bugs begin in data handling, not in the model itself.

In [ ]:
sample_text = "Hello, World!"
sample_tokens = encode(sample_text)
round_trip = decode(sample_tokens)
print(sample_tokens)
print(round_trip)
assert round_trip == sample_text

train_data, val_data = prepare_data()
print(f"Total tokens: {len(train_data) + len(val_data):,}")
print(f"Train tokens: {len(train_data):,}")
print(f"Val tokens:   {len(val_data):,}")

x, y = get_batch(train_data, batch_size=4, seq_len=32)
print(f"Batch shapes: x={x.shape}, y={y.shape}")
print(f"First input preview:  {decode(x[0].tolist())!r}")
print(f"First target preview: {decode(y[0].tolist())!r}")
assert torch.equal(x[0, 1:], y[0, :-1])

## 10. Training Utilities

The next few functions make the training loop easier to read.
They are the same as in the simple transformer notebook, with one key difference: the training loop logs the auxiliary load-balancing loss alongside the main cross-entropy loss so you can monitor expert utilization during training.

### `get_lr`

A two-stage learning-rate schedule: linear warmup followed by cosine decay.
Starting small keeps early updates stable; gradually lowering the rate helps the model settle into a better solution.

In [ ]:
def get_lr(step: int, max_lr: float, min_lr: float, warmup_steps: int, total_steps: int) -> float:
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps

    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    cosine = 0.5 * (1 + math.cos(math.pi * progress))
    return min_lr + (max_lr - min_lr) * cosine

### `estimate_loss`

Training loss on a single minibatch is noisy.
`estimate_loss` gives a more trustworthy picture by averaging over several fresh batches from both the training split and the validation split.

In [ ]:
@torch.no_grad()
def estimate_loss(model, train_data, val_data, batch_size, seq_len, device, eval_iters=20):
    model.eval()
    results = {}
    for name, data in [("train", train_data), ("val", val_data)]:
        losses = []
        for _ in range(eval_iters):
            x, y = get_batch(data, batch_size, seq_len, device)
            _, loss = model(x, y)
            losses.append(loss.item())
        results[name] = sum(losses) / len(losses)
    model.train()
    return results

### `save_checkpoint` and `load_checkpoint`

A checkpoint captures the current training state so you can pause work, resume later, or keep the best-performing model.
We save the model weights, optimizer state, config, and current step.

Note that `load_checkpoint` rebuilds an `MoEModel` (not a `GPT`) from the saved config.

In [ ]:
def save_checkpoint(model, config, optimizer, step, path):
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "config": config,
            "step": step,
        },
        path,
    )


def load_checkpoint(path, device="cpu"):
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    config = checkpoint["config"]
    model = MoEModel(config).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    return model, config, checkpoint["step"]

### `get_device`

This helper picks the best available accelerator: CUDA, then Apple MPS, then CPU.

In [ ]:
def get_device():
    if torch.cuda.is_available():
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"

### `sanity_check`

Before a long training run, overfit a single fixed batch.
If the loss does not fall very close to zero, the problem is usually fundamental: a bug in the model, data pipeline, or optimization loop.

This is one of the most useful debugging tricks in deep learning.

In [ ]:
def sanity_check(config, device):
    print("=" * 60)
    print("SANITY CHECK: overfitting one batch")
    print("=" * 60)

    model = MoEModel(config).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
    train_data, _ = prepare_data()
    x, y = get_batch(train_data, batch_size=4, seq_len=config.seq_len, device=device)

    for step in range(500):
        _, loss = model(x, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step % 50 == 0:
            print(f"step {step:4d} | loss {loss.item():.4f}")

    final_loss = loss.item()
    print(f"Final loss: {final_loss:.4f}")
    return final_loss < 0.1

### `train`

This is the full training loop.
It combines every previous idea: data loading, model creation, optimizer setup, learning-rate scheduling, gradient computation, gradient clipping, validation checks, and checkpoint saving.

**MoE-specific detail in the training loop:**

The loss logged at each step includes both the cross-entropy loss and the auxiliary load-balancing loss (weighted by `aux_loss_weight=0.01`).
As training progresses, you should see:
- The total loss decreasing as the model learns to predict text.
- The auxiliary loss contribution staying small, which means the load balancer is working and experts are being utilized roughly equally.

If the auxiliary loss grows large, it usually means the router is collapsing and you may need to increase `aux_loss_weight`.

In [ ]:
def train(
    config,
    device,
    max_steps=2000,
    batch_size=32,
    max_lr=3e-3,
    min_lr=3e-4,
    warmup_steps=100,
    eval_interval=100,
    save_interval=500,
    checkpoint_dir=CHECKPOINT_DIR,
):
    print("=" * 60)
    print("TRAINING")
    print("=" * 60)
    print(f"Device: {device}")
    print(f"Config: n_embd={config.n_embd}, n_head={config.n_head}, n_layer={config.n_layer}")
    print(f"  Dense layers: {config.n_dense_layers}, MoE layers: {config.n_layer - config.n_dense_layers}")
    print(f"  Experts: {config.n_experts} total, {config.n_experts_active} active per token")
    print(f"  Shared expert: {config.has_shared_expert}")
    print(f"  Aux loss weight: {config.aux_loss_weight}")
    print(f"Steps: {max_steps}, Batch size: {batch_size}, Seq len: {config.seq_len}")
    print(f"LR: {max_lr} -> {min_lr} (warmup {warmup_steps} steps)")
    print()

    train_data, val_data = prepare_data()
    model = MoEModel(config).to(device)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {total_params:,}")

    decay_params = [p for p in model.parameters() if p.dim() >= 2]
    nodecay_params = [p for p in model.parameters() if p.dim() < 2]
    optimizer = torch.optim.AdamW(
        [
            {"params": decay_params, "weight_decay": 0.1},
            {"params": nodecay_params, "weight_decay": 0.0},
        ],
        lr=max_lr,
        betas=(0.9, 0.95),
    )

    use_amp = device in ("cuda", "mps")
    scaler = torch.amp.GradScaler(enabled=(device == "cuda"))
    amp_dtype = torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported() else torch.float16

    os.makedirs(checkpoint_dir, exist_ok=True)
    best_val_loss = float("inf")
    t0 = time.time()

    for step in range(max_steps):
        lr = get_lr(step, max_lr, min_lr, warmup_steps, max_steps)
        for param_group in optimizer.param_groups:
            param_group["lr"] = lr

        x, y = get_batch(train_data, batch_size, config.seq_len, device)

        if use_amp:
            with torch.autocast(device_type=device, dtype=amp_dtype):
                _, loss = model(x, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            _, loss = model(x, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        optimizer.zero_grad(set_to_none=True)

        if step % 10 == 0:
            elapsed = time.time() - t0
            tokens_per_sec = (step + 1) * batch_size * config.seq_len / elapsed if elapsed > 0 else 0
            print(f"step {step:5d} | loss {loss.item():.4f} (CE + aux) | lr {lr:.2e} | {tokens_per_sec:.0f} tok/s")

        if step > 0 and step % eval_interval == 0:
            losses = estimate_loss(model, train_data, val_data, batch_size, config.seq_len, device)
            print(f"eval | train {losses['train']:.4f} | val {losses['val']:.4f}")
            if losses["val"] < best_val_loss:
                best_val_loss = losses["val"]
                save_checkpoint(model, config, optimizer, step, Path(checkpoint_dir) / "best.pt")
                print("saved new best checkpoint")

        if step > 0 and step % save_interval == 0:
            save_checkpoint(model, config, optimizer, step, Path(checkpoint_dir) / "latest.pt")

    save_checkpoint(model, config, optimizer, max_steps, Path(checkpoint_dir) / "final.pt")
    print(f"Training complete. Best val loss: {best_val_loss:.4f}")
    return model

## 11. Lightweight Pipeline Preview

Instead of launching a full training run right away, this cell does a quick end-to-end pass:
- choose a device,
- build the config and model,
- load data,
- sample one batch,
- compute one forward pass.

This catches integration problems early while staying fast enough for iteration.

In [ ]:
device = get_device()
config = MoEConfig()
model = MoEModel(config).to(device)
train_data, val_data = prepare_data()
x, y = get_batch(train_data, batch_size=2, seq_len=32, device=device)
logits, loss = model(x, y)

print(f"Selected device: {device}")
print(f"Input batch shape:  {x.shape}")
print(f"Target batch shape: {y.shape}")
print(f"Logits shape:       {logits.shape}")
print(f"One-batch loss:     {loss.item():.4f} (includes CE + aux loss)")

## 12. Text Generation

Training teaches the model a probability distribution over the next token.
Generation turns that distribution into actual text by repeating the same loop:
1. Feed the current context into the model.
2. Read the logits for the last position.
3. Choose the next token.
4. Append it to the context.
5. Repeat.

Temperature and top-k are two simple ways to control the sampling behavior.

### `generate`

This function implements autoregressive sampling.

What each argument teaches:
- `prompt_tokens`: generation always starts from some context.
- `max_new_tokens`: how many steps to take.
- `temperature`: lower values make the model more conservative; higher values make it more random.
- `top_k`: restricts sampling to the most plausible candidates.

A subtle but important detail:
we only feed the last `seq_len` tokens back into the model, because the model was trained with a fixed maximum context window.

In [ ]:
@torch.no_grad()
def generate(model, prompt_tokens: list[int], max_new_tokens: int = 200, temperature: float = 0.8, top_k: int = 50, device: str = "cpu") -> list[int]:
    model.eval()
    seq_len = model.config.seq_len
    tokens = torch.tensor([prompt_tokens], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        idx = tokens[:, -seq_len:]
        logits, _ = model(idx)
        logits = logits[:, -1, :]

        if temperature == 0:
            next_token = logits.argmax(dim=-1, keepdim=True)
        else:
            logits = logits / temperature
            if top_k > 0:
                top_values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < top_values[:, -1:]] = float("-inf")
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

        tokens = torch.cat([tokens, next_token], dim=1)

    return tokens[0].tolist()

## 13. Optional Checkpoint Demo

This cell tries to load a saved checkpoint and generate text.
If no checkpoint exists yet, it prints a friendly message instead of failing.

That makes the notebook safe to run top-to-bottom in a fresh clone while still showing how inference works once you have trained the model.

In [ ]:
checkpoint_path = CHECKPOINT_DIR / "best.pt"

if checkpoint_path.exists():
    demo_model, demo_config, step = load_checkpoint(checkpoint_path, device)
    prompt_tokens = encode("ROMEO:")
    output_tokens = generate(
        demo_model,
        prompt_tokens,
        max_new_tokens=200,
        temperature=0.8,
        top_k=50,
        device=device,
    )
    print(f"Loaded checkpoint from step {step}")
    print(decode(output_tokens))
else:
    print(f"No checkpoint found at {checkpoint_path}.")
    print("Run a training cell first if you want to try generation from learned weights.")

## 14. Exercises

Try these before reading the scaffold cell below:

1. **Change the number of experts.** Set `n_experts=4` and `n_experts_active=1`. Build the model and compare total vs active parameters to the default config. How does the sparsity ratio change?

2. **Disable the shared expert.** Set `has_shared_expert=False`. Build the model and compare the parameter count. Think about what patterns might be lost without a shared expert.

3. **Increase the dense prefix.** Set `n_dense_layers=3` (out of 4 total). This makes most of the model dense. Compare the total and active parameter counts. When would you want more dense layers vs more MoE layers?

4. **Thought experiment.** DeepSeek V3 has 256 experts with top-2 routing. If you doubled the number of experts to 512 (keeping top-2), what happens to total parameters? What happens to active parameters? What happens to training compute?

Reflection question:
Why do you think MoE models are popular for very large models but less common for small ones?

In [ ]:
# Exercise 1: Fewer experts, top-1 routing
print("=" * 60)
print("Exercise 1: n_experts=4, n_experts_active=1")
print("=" * 60)
exercise1_config = MoEConfig(n_experts=4, n_experts_active=1)
exercise1_model = MoEModel(exercise1_config)
exercise1_model.count_parameters()

print()

# Exercise 2: No shared expert
print("=" * 60)
print("Exercise 2: has_shared_expert=False")
print("=" * 60)
exercise2_config = MoEConfig(has_shared_expert=False)
exercise2_model = MoEModel(exercise2_config)
exercise2_model.count_parameters()

print()

# Exercise 3: Mostly dense
print("=" * 60)
print("Exercise 3: n_dense_layers=3 (mostly dense)")
print("=" * 60)
exercise3_config = MoEConfig(n_dense_layers=3)
exercise3_model = MoEModel(exercise3_config)
exercise3_model.count_parameters()

print()

# Compare all configs
print("=" * 60)
print("Default config for comparison:")
print("=" * 60)
default_model = MoEModel(MoEConfig())
default_model.count_parameters()

## 15. Common Pitfalls and Next Steps

### Common Pitfalls
- **Running cells out of order**: later function cells depend on earlier imports and class definitions.
- **Router collapse**: if you set `aux_loss_weight` too low (or to 0), the router will send all tokens to 1-2 experts and most of your model capacity is wasted. Monitor the loss and check that it stays reasonable.
- **Confusing total and active parameters**: a 671B parameter MoE model does NOT use 671B parameters per token. Only the active parameters (37B in DeepSeek V3) determine per-token compute cost.
- **Expert loop inefficiency**: our loop-over-experts implementation is simple but slow. Production MoE models use grouped GEMM or expert parallelism across GPUs.
- **Using a sequence length longer than the configured context window**: the model will assert if `T > seq_len`.
- **Expecting identical speed to dense models**: even though active parameters are fewer, the routing overhead and memory for all expert weights means MoE models are not as fast as a dense model with the same active parameter count.
- **Judging learning from one noisy batch**: use `estimate_loss` instead of trusting a single minibatch.

### Extensions
- Add expert utilization logging to the training loop (track how many tokens go to each expert per batch).
- Implement expert capacity: cap the number of tokens each expert can process to prevent overflow.
- Try different values of `aux_loss_weight` and observe the effect on expert balance.
- Replace the loop-over-experts with a batched implementation for better GPU utilization.
- Add expert dropout: randomly drop experts during training to improve robustness.
- Experiment with different numbers of dense prefix layers and measure the effect on quality.
- Compare training curves between a dense model and an MoE model with the same active parameter count.